# 15.08 - Tiny diffusion model training

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** Tiny noise-prediction model and reconstruction evidence.

Train a compact PyTorch noise predictor on deterministic 8×8 images, using random timesteps and the standard noise-prediction MSE objective.

## Core Ideas

Diffusion training samples a timestep, constructs a noisy image from cumulative alpha, and teaches a network to predict the injected noise. Inputs are commonly scaled to `[-1,1]`. A timestep channel gives the network noise-level context. Fixed validation noise makes reconstruction evidence comparable.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 15
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepared Tiny Images and Schedule

Thirty-two 8×8 patterns are already scaled to `[-1,1]`. The linear schedule contains ten timesteps.

In [ ]:
clean_images = torch.full((32, 1, 8, 8), -1.0)
for index in range(32):
    offset = 1 + index % 4
    clean_images[index, 0, offset:offset + 3, 2:6] = 1.0
train_images, validation_images = clean_images[:24], clean_images[24:]
betas = torch.linspace(0.0001, 0.08, 10)
alpha_bars = torch.cumprod(1.0 - betas, dim=0)
fixed_validation_noise = torch.randn_like(validation_images)
print("train/validation/schedule:", train_images.shape, validation_images.shape, alpha_bars.shape)

## Exercise 15-A: Construct noisy training inputs

Sample one timestep per image and broadcast cumulative-alpha coefficients across spatial dimensions.

**Return structure — `sample_noisy_batch`:** A tuple `(noisy, noise, timesteps)`: float32 tensors `noisy` and `noise` share `[N,1,8,8]`; `timesteps` is int64 `[N]`, all on `device`.

In [ ]:
def sample_noisy_batch(clean, schedule, device=DEVICE):
    clean = clean.to(device)
    schedule = schedule.to(device)
    timesteps = torch.randint(0, len(schedule), (len(clean),), device=device)
    noise = torch.randn_like(clean)
    selected = schedule[timesteps].reshape(-1, 1, 1, 1)
    noisy = selected.sqrt() * clean + (1.0 - selected).sqrt() * noise
    return noisy, noise, timesteps


# Smoke check: sample one complete training batch.
noisy_batch, sampled_noise, sampled_timesteps = sample_noisy_batch(train_images, alpha_bars)
print(noisy_batch.shape, sampled_timesteps[:8])

## Exercise 15-B: Build a timestep-conditioned predictor

Concatenate a normalized timestep channel with the noisy image and predict one noise channel.

**Return structure — `build_noise_predictor`:** An `nn.Module` on `device` that maps float tensors `[N,2,8,8]` to predicted noise `[N,1,8,8]`.

In [ ]:
def build_noise_predictor(device=DEVICE):
    return nn.Sequential(nn.Conv2d(2, 12, 3, padding=1), nn.ReLU(), nn.Conv2d(12, 12, 3, padding=1), nn.ReLU(), nn.Conv2d(12, 1, 3, padding=1)).to(device)


# Smoke check: verify the model boundary.
noise_predictor = build_noise_predictor()
timestep_channel = sampled_timesteps.float().reshape(-1, 1, 1, 1).expand(-1, 1, 8, 8) / 9.0
print(noise_predictor(torch.cat([noisy_batch, timestep_channel], dim=1)).shape)

## Exercise 15-C: Train the noise predictor

Use the complete training split for every step and record prediction MSE.

**Return structure — `train_noise_predictor`:** A `list[dict]` of length `steps`; every row has integer `step` and Python float `noise_mse`. The model is updated.

In [ ]:
def train_noise_predictor(model, images, schedule, steps=20, device=DEVICE):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    history = []
    for step in range(1, steps + 1):
        noisy, noise, timesteps = sample_noisy_batch(images, schedule, device)
        time_channel = timesteps.float().reshape(-1, 1, 1, 1).expand(-1, 1, noisy.shape[2], noisy.shape[3]) / max(len(schedule) - 1, 1)
        optimizer.zero_grad()
        predicted = model(torch.cat([noisy, time_channel], dim=1))
        loss = nn.functional.mse_loss(predicted, noise)
        loss.backward(); optimizer.step()
        history.append({"step": step, "noise_mse": float(loss.detach().cpu())})
    return history


# Smoke check and complete training-split evidence.
diffusion_history = train_noise_predictor(noise_predictor, train_images, alpha_bars)
print("first/last:", diffusion_history[0], diffusion_history[-1])

## Exercise 15-D: Evaluate fixed-noise reconstruction

Noise validation images at the final timestep, predict fixed noise, reconstruct clean estimates, and report both errors.

**Return structure — `evaluate_noise_predictor`:** A dictionary with CPU float32 `reconstructions` `[N,1,8,8]` and Python floats `noise_mse` and `reconstruction_mse`.

In [ ]:
def evaluate_noise_predictor(model, clean, noise, schedule, device=DEVICE):
    model.eval(); clean, noise, schedule = clean.to(device), noise.to(device), schedule.to(device)
    timestep = len(schedule) - 1; alpha_bar = schedule[timestep]
    noisy = alpha_bar.sqrt() * clean + (1.0 - alpha_bar).sqrt() * noise
    time_channel = torch.full_like(noisy, float(timestep) / max(len(schedule) - 1, 1))
    with torch.inference_mode():
        predicted_noise = model(torch.cat([noisy, time_channel], dim=1))
    reconstructions = (noisy - (1.0 - alpha_bar).sqrt() * predicted_noise) / alpha_bar.sqrt()
    return {"reconstructions": reconstructions.detach().cpu(), "noise_mse": float(nn.functional.mse_loss(predicted_noise, noise).cpu()), "reconstruction_mse": float(nn.functional.mse_loss(reconstructions, clean).cpu())}


# Smoke check: evaluate every validation image with fixed noise.
diffusion_validation = evaluate_noise_predictor(noise_predictor, validation_images, fixed_validation_noise, alpha_bars)
print({key: value for key, value in diffusion_validation.items() if key != "reconstructions"})

## Test Cases

**Return structure — `run_day15_tests`:** Returns `None`; assertions and `Day 15 tests passed` communicate success.

In [ ]:
def run_day15_tests():
    assert noisy_batch.shape == sampled_noise.shape == train_images.shape
    assert sampled_timesteps.shape == (24,) and sampled_timesteps.dtype == torch.int64
    assert noise_predictor(torch.cat([noisy_batch, timestep_channel], dim=1)).shape == train_images.shape
    assert len(diffusion_history) == 20 and set(diffusion_history[0]) == {"step", "noise_mse"}
    assert diffusion_validation["reconstructions"].shape == validation_images.shape
    assert diffusion_validation["noise_mse"] >= 0 and diffusion_validation["reconstruction_mse"] >= 0
    print("Day 15 tests passed")


run_day15_tests()

## Day 15 Checklist

- [ ] Scale images to the model's documented range.
- [ ] Broadcast cumulative-alpha coefficients correctly.
- [ ] Condition the predictor on timestep.
- [ ] Evaluate with fixed validation noise.
- [ ] Run the test cases.